Purpose
-------
This notebook converts CIDOC CRM-based Turtle files into document-level JSON
records for information extraction evaluation.

The pipeline:
1. Loads RDF/Turtle files from Google Drive.
2. Extracts people, places, events, relations, and attributes from RDF graphs.
3. Normalizes the extracted data into a compact JSON schema.
4. Completes the entity inventory with NER annotations from the original dataset when attested person or place names are not already represented in the Turtle graph.
5. Preserves entity names in their attested form; no authority normalization or cross-document entity reconciliation is performed.
6. Saves one JSON file per document and one corpus-level JSONL file.

The output is designed for evaluating structured information extraction over
medieval documentary records.

In [ ]:
!pip -q install rdflib pandas tqdm

# Configuration

This section installs the required dependencies, mounts Google Drive, and defines
the input and output paths used by the corpus generation pipeline.

Expected inputs:
- Turtle files containing RDF representations of the source documents.
- An optional NER JSON file with detected person and place mentions.

Generated outputs:
- One document-level JSON file per document.
- A corpus-level JSONL file containing all processed documents.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
from collections import defaultdict
from dataclasses import dataclass

from tqdm.auto import tqdm
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import RDF

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path("/content/drive/MyDrive/Registros medievales")

TTL_DIR = BASE_DIR / "ttl_attested"

JSON_DIR = BASE_DIR / "json_ie_eval"
JSON_DIR.mkdir(exist_ok=True)

JSONL_PATH = BASE_DIR / "all_documents_ie_eval.jsonl"

NER_PATH = BASE_DIR / "NER/ner.json"

print("TTL_DIR =", TTL_DIR)
print("JSON_DIR =", JSON_DIR)
print("JSONL_PATH =", JSONL_PATH)
print("NER_PATH  =", NER_PATH, "(exists:", NER_PATH.exists(), ")")

# General RDF and text utilities

This section defines reusable helper functions for parsing Turtle files,
navigating RDF graphs, extracting literals and URI references, normalizing text,
deduplicating values, and loading external NER annotations.

These functions are intentionally schema-light: most RDF access is performed
through local predicate names so that the pipeline remains robust to namespace
variation across Turtle files.

In [ ]:
def parse_ttl(path):
    """Parse a Turtle file into an RDFLib graph."""
    g = Graph()
    g.parse(path, format="turtle")
    return g


def local_name(term):
    """Return the local fragment of a URI, RDF term, or literal."""
    if isinstance(term, Literal):
        return str(term)

    s = str(term)

    if "#" in s:
        return s.rsplit("#", 1)[-1]

    return s.rsplit("/", 1)[-1]


def get_objects(g, subj, pred_local_name):
    """Return all objects connected to a subject by a predicate local name."""
    objs = []

    for s, p, o in g.triples((subj, None, None)):
        if local_name(p) == pred_local_name:
            objs.append(o)

    return objs


def get_object_literals(g, subj, pred_local_name):
    """Return literal object values for a subject and predicate local name."""
    return [
        str(o)
        for o in get_objects(g, subj, pred_local_name)
        if isinstance(o, Literal)
    ]


def get_object_uris(g, subj, pred_local_name):
    """Return URI object values for a subject and predicate local name."""
    return [
        o
        for o in get_objects(g, subj, pred_local_name)
        if isinstance(o, URIRef)
    ]


def get_first_literal(g, subj, pred_local_name, default=None):
    """Return the first literal object for a predicate, or a default value."""
    vals = get_object_literals(g, subj, pred_local_name)
    return vals[0] if vals else default


def get_first_uri(g, subj, pred_local_name, default=None):
    """Return the first URI object for a predicate, or a default value."""
    vals = get_object_uris(g, subj, pred_local_name)
    return vals[0] if vals else default


def get_types(g, subj):
    """Return the RDF type local names associated with a subject."""
    return [
        local_name(o)
        for o in get_objects(g, subj, local_name(RDF.type))
    ]


def subjects_of_type(g, type_local_name):
    """Return all subjects whose RDF type matches the requested local name."""
    out = []

    for s, p, o in g:
        if p == RDF.type and local_name(o) == type_local_name:
            out.append(s)

    return sorted(set(out), key=str)


def uri_to_id(uri):
    """Convert a URI reference into its local identifier."""
    return local_name(uri) if uri is not None else None


def unique_preserve_order(values):
    """Remove duplicate values while preserving their original order."""
    seen = set()
    out = []

    for v in values:
        key = (
            json.dumps(v, sort_keys=True, ensure_ascii=False)
            if isinstance(v, (dict, list))
            else str(v)
        )

        if key not in seen:
            seen.add(key)
            out.append(v)

    return out


def get_label_from_node(g, subj):
    """Extract the main human-readable label from a complex RDF node."""
    return get_first_literal(g, subj, "P1_is_identified_by")


def get_linked_ids(g, subj, pred_local_name):
    """Return local identifiers for URI objects linked through a predicate."""
    return [
        uri_to_id(o)
        for o in get_object_uris(g, subj, pred_local_name)
    ]


def get_single_linked_id(g, subj, pred_local_name):
    """Return the first linked local identifier for a predicate."""
    vals = get_linked_ids(g, subj, pred_local_name)
    return vals[0] if vals else None


def get_linked_ids_by_predicate_name(g, subj, pred_local_name):
    """Return linked local identifiers for an exact predicate local name."""
    vals = []

    for s, p, o in g.triples((subj, None, None)):
        if local_name(p) == pred_local_name and isinstance(o, URIRef):
            vals.append(uri_to_id(o))

    return unique_preserve_order(vals)


def get_event_type_labels(g, subj):
    """Extract human-readable event type labels from P2_has_type nodes."""
    labels = []

    for obj in get_object_uris(g, subj, "P2_has_type"):
        label = get_label_from_node(g, obj)

        if label:
            labels.append(label)

    return unique_preserve_order(labels)


def get_time_values(g, subj):
    """Extract time values from CIDOC CRM time-span nodes."""
    values = []

    for ts in get_object_uris(g, subj, "P4_has_time-span"):

        v = get_first_literal(g, ts, "P82_at_some_time_within")

        if v:
            values.append(v)
            continue

        for s, p, o in list(g.triples((ts, None, None))):
            if isinstance(o, Literal):
                values.append(str(o))

    return unique_preserve_order(values)


def get_price_values(g, subj):
    """Extract structured price values and currencies from event price nodes."""
    prices = []

    for amount in get_object_uris(g, subj, "P179_had_sales_price"):

        value = get_first_literal(g, amount, "P90_has_value")

        currency_label = None

        currency_node = get_first_uri(g, amount, "P180_has_currency")

        if currency_node is not None:
            currency_label = get_label_from_node(g, currency_node)

        price_obj = {}

        if value is not None:
            try:
                price_obj["value"] = float(value)
            except Exception:
                price_obj["value"] = value

        if currency_label:
            price_obj["currency"] = currency_label

        if price_obj:
            prices.append(price_obj)

    return unique_preserve_order(prices)


def get_witnesses(g, subj):
    """Return participant identifiers when the event role is labelled as witness."""
    if not has_role_label(g, subj, "witness"):
        return []

    return unique_preserve_order(
        get_linked_ids(g, subj, "P14_carried_out_by")
    )


def strip_accents(s):
    """Remove diacritical marks from a Unicode string."""
    return "".join(
        ch
        for ch in unicodedata.normalize("NFD", s)
        if unicodedata.category(ch) != "Mn"
    )


def normalize_text(s):
    """
    Apply lightweight string normalization for duplicate avoidance and lookup.

    Normalization is used only for conservative intra-document matching between
    attested entity forms and NER-derived entity forms.
    """
    if s is None:
        return ""

    s = str(s).strip().lower()
    s = strip_accents(s)
    s = s.replace("_", " ")

    s = re.sub(r"['´`]", "", s)
    s = re.sub(r"[^a-z0-9\s-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s


def safe_join(parts):
    """Join non-empty string components with single spaces."""
    return " ".join([
        p.strip()
        for p in parts
        if p and str(p).strip()
    ])


def load_ner_json(path):
    """Load external NER annotations from JSON."""
    if path is None or not Path(path).exists():
        return {}

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
# --- Event and role helpers ---

def get_role_labels(g, subj):
    """Extract role labels associated with an event participant relation."""
    labels = []

    for role_node in get_object_uris(g, subj, "P14.1_in_the_role_of"):
        label = get_label_from_node(g, role_node)

        if label:
            labels.append(label)

    return unique_preserve_order(labels)


def has_role_label(g, subj, target_label):
    """Check whether an event contains a specific role label."""
    target = target_label.lower()

    return any(
        label.lower() == target
        for label in get_role_labels(g, subj)
    )


def init_raw_event(g, subj, event_type):
    """Create a raw event object and attach common type and time metadata."""
    event = {
        "source_id": uri_to_id(subj),
        "rdf_types": get_types(g, subj),
        "type": event_type,
        "subtype": None,
        "attributes": {},
        "arguments": [],
        "_warnings": []
    }

    type_labels = get_event_type_labels(g, subj)

    if type_labels:
        event["subtype"] = type_labels[0]

        if len(type_labels) > 1:
            event["attributes"]["event_type_labels"] = type_labels

    else:
        event["_warnings"].append("No P2_has_type label found")

    time_values = get_time_values(g, subj)

    if time_values:
        event["attributes"]["time"] = time_values

    else:
        event["_warnings"].append("No time value found")

    return event


def add_event_arguments(event, role, source_ids):
    """Append document-level role-bearing source entity identifiers to a raw event."""
    for sid in unique_preserve_order(source_ids):

        event["arguments"].append({
            "role": role,
            "source_entity_id": sid
        })


def finalize_raw_event(event):
    """Finalize a raw event by validating arguments and pruning empty warnings."""
    if not event["arguments"]:
        event["_warnings"].append("No event arguments found")

    if not event["_warnings"]:
        del event["_warnings"]

    return event

# Schema normalization

This section defines the normalized schema used by the corpus generator.

It includes:
- Field lists for person attributes and relations.
- Predicate mappings from source RDF fields to normalized JSON predicates.
- Event type mappings from RDF classes to canonical event categories.
- NER label mappings used to import attested person and place entity forms from the original dataset.
- A counter class used to generate deterministic document-local identifiers.
- Helper functions for canonical names, aliases, object typing, and event typing.

In [ ]:
PERSON_LITERAL_FIELDS = [
    "givenName",
    "familyName",
    "nickname",
]

PERSON_SINGLE_LINK_FIELDS = [
    "residence",
    "spouse",
    "father",
    "mother",
    "employer",
    "relative_grandfather",
    "relative_grandmother",
    "relative_sibling-in-law",
    "relative_father-in-law",
    "relative_son-in-law",
]
PERSON_MULTI_LINK_FIELDS = [
    "child",
    "sibling",
    "relative_cousin",
    "relative_uncle",
    "relative_nephew",
    "relative_grandchild",
]

PREDICATE_MAP = {
    "givenName": "given_name",
    "familyName": "family_name",
    "nickname": "nickname",
    "occupation": "occupation",
    "position_held": "position_held",
    "occupation_location": "occupation_location",
    "position_jurisdiction": "position_jurisdiction",
    "residence": "residence",
    "spouse": "has_spouse",
    "father": "has_father",
    "mother": "has_mother",
    "employer": "employer",
    "relative_grandfather": "has_grandfather",
    "relative_grandmother": "has_grandmother",
    "relative_sibling-in-law": "has_sibling_in_law",
    "relative_father-in-law": "has_father_in_law",
    "relative_son-in-law": "has_son_in_law",
    "child": "has_child",
    "sibling": "has_sibling",
    "relative_cousin": "has_cousin",
    "relative_uncle": "has_uncle",
    "relative_nephew": "has_nephew",
    "relative_grandchild": "has_grandchild",
}

TRANSFER_EVENT_TYPES = [
    "E96_Purchase",
    "E8_Acquisition",
]

EVENT_TYPE_MAP = {
    "E96_Purchase": "Purchase",
    "E8_Acquisition": "Acquisition",
    "E12_Production": "DocumentProduction",
    "E7_Activity": "GenericActivity",
}

NER_LABEL_TO_ENTITY_TYPE = {
    "PERS": "Person",
    "LOC": "Place",
}

In [ ]:
@dataclass
class Counters:
    """
    Maintain deterministic counters for normalized document-local identifiers.

    The class generates sequential identifiers for:
    - Entities.
    - Facts.
    - Events.

    These identifiers are stable within a document transformation run and are
    used throughout the normalized JSON representation.
    """

    entity: int = 0
    fact: int = 0
    event: int = 0

    def next_entity_id(self, entity_type):
        """
        Generate the next document-level entity identifier.

        Parameters
        ----------
        entity_type : str
            Canonical entity type name.

        Returns
        -------
        str
            Deterministic entity identifier.
        """
        self.entity += 1
        return f"ent_{entity_type.lower()}_{self.entity:04d}"

    def next_fact_id(self):
        """
        Generate the next document-level fact identifier.

        Returns
        -------
        str
            Deterministic fact identifier.
        """
        self.fact += 1
        return f"fact_{self.fact:05d}"

    def next_event_id(self):
        """
        Generate the next document-level event identifier.

        Returns
        -------
        str
            Deterministic event identifier.
        """
        self.event += 1
        return f"evt_{self.event:04d}"

In [ ]:
def choose_canonical_name(entity_type, raw):
    """
    Select the preferred document-level display name for an entity.

    The returned value preserves attested naming forms from the source Turtle
    data and is used only as a stable display label during corpus construction.

    For people, the function prioritizes given name plus family name, then
    nickname, and finally the source identifier. For places, it prioritizes the
    first available label and falls back to the source identifier.

    The ``canonical_name`` field does not imply authority normalization,
    cross-document reconciliation, or external entity linking.
    """
    if entity_type == "Person":
        given = raw.get("givenName", [])
        family = raw.get("familyName", [])
        nickname = raw.get("nickname", [])

        full_name = safe_join([
            given[0] if given else "",
            family[0] if family else "",
        ])

        if full_name:
            return full_name

        if nickname:
            return nickname[0]

        return raw["source_id"]

    elif entity_type == "Place":
        labels = raw.get("labels", [])
        return labels[0] if labels else raw["source_id"]

    return raw["source_id"]


def build_aliases(entity_type, raw):
    """
    Build lightweight lookup aliases for duplicate avoidance.

    Aliases are derived from attested entity forms and are used only for
    intra-document matching when importing NER-derived entities.

    For places, aliases may additionally include URI-derived fallback forms used
    for conservative duplicate avoidance.
    """
    aliases = []

    if entity_type == "Person":
        given = raw.get("givenName", [])
        family = raw.get("familyName", [])
        occupations = raw.get("occupation", [])
        nicknames = raw.get("nickname", [])

        given_0 = given[0] if given else ""
        family_0 = family[0] if family else ""
        occupation_0 = occupations[0] if occupations else ""
        nickname_0 = nicknames[0] if nicknames else ""

        candidates = [
            safe_join([given_0, family_0, occupation_0, nickname_0]),
            safe_join([given_0, family_0, nickname_0]),
            safe_join([given_0, family_0, occupation_0]),
            safe_join([given_0, family_0]),
        ]

        aliases.extend([c for c in candidates if c])

    elif entity_type == "Place":
        labels = raw.get("labels", [])

        if labels:
            aliases.append(labels[0])

        aliases.append(raw["source_id"].replace("_", " "))

    aliases = [
        a
        for a in aliases
        if a and str(a).strip()
    ]

    seen = set()
    out = []

    for a in aliases:
        key = normalize_text(a)

        if key and key not in seen:
            seen.add(key)
            out.append(a)

    return out


def object_type_for_predicate_object(obj):
    """
    Classify a document-level fact object as an entity reference or literal value.

    Objects whose identifiers start with ``ent_`` are treated as normalized
    entity references. All other objects are treated as literal values.
    """
    return (
        "entity"
        if isinstance(obj, str) and obj.startswith("ent_")
        else "literal"
    )


def choose_event_type_from_rdf_types(rdf_types):
    """
    Select the canonical event type from the RDF type hierarchy.

    Event types are selected according to a fixed priority order:
    1. E96_Purchase -> Purchase
    2. E8_Acquisition -> Acquisition
    3. E12_Production -> DocumentProduction
    4. E7_Activity -> GenericActivity

    Returns ``None`` when no supported event type is found.
    """
    priority = [
        "E96_Purchase",
        "E8_Acquisition",
        "E12_Production",
        "E7_Activity",
    ]

    rdf_types = rdf_types or []

    for rdf_type in priority:
        if rdf_type in rdf_types:
            return EVENT_TYPE_MAP[rdf_type]

    return None

# Raw extraction from RDF

This section extracts source-level records from each RDF graph before final JSON
normalization.

The extraction layer preserves source identifiers and RDF-derived values for:
- People.
- Places.
- Events.

Warnings are collected when expected labels, times, arguments, or linked
entities are missing.

In [ ]:
def extract_raw_person(g, subj):
    """
    Extract a raw person record from an RDF subject.

    The raw record preserves the source identifier, RDF types, literal name
    fields, occupations, positions, associated places, and person-to-person
    relations. Missing labels in complex occupation or position nodes are stored
    as warnings.
    """
    person = {
        "source_id": uri_to_id(subj),
        "rdf_types": get_types(g, subj),
        "_warnings": []
    }

    # Literal attributes.
    for field in PERSON_LITERAL_FIELDS:
        values = unique_preserve_order(
            get_object_literals(g, subj, field)
        )

        if values:
            person[field] = values

    # Occupation labels and associated work locations.
    occupation_values = []
    occupation_locations = []

    for obj in get_object_uris(g, subj, "occupation"):
        label = get_label_from_node(g, obj)

        if label:
            occupation_values.append(label)
        else:
            person["_warnings"].append(
                f"occupation node {uri_to_id(obj)} has no P1_is_identified_by"
            )

        for place in get_object_uris(g, obj, "work_location"):
            occupation_locations.append(uri_to_id(place))

    # Compatibility fallback for models where work_location is attached directly
    # to the person rather than to the occupation node.
    for place in get_object_uris(g, subj, "work_location"):
        occupation_locations.append(uri_to_id(place))

    occupation_values = unique_preserve_order(occupation_values)
    occupation_locations = unique_preserve_order(occupation_locations)

    if occupation_values:
        person["occupation"] = occupation_values

    if occupation_locations:
        person["occupation_location"] = occupation_locations

    # Position labels and associated jurisdictions.
    position_values = []
    position_jurisdictions = []

    for obj in get_object_uris(g, subj, "position_held"):
        label = get_label_from_node(g, obj)

        if label:
            position_values.append(label)
        else:
            person["_warnings"].append(
                f"position_held node {uri_to_id(obj)} has no P1_is_identified_by"
            )

        for place in get_object_uris(g, obj, "applies_to_jurisdiction"):
            position_jurisdictions.append(uri_to_id(place))

    position_values = unique_preserve_order(position_values)
    position_jurisdictions = unique_preserve_order(position_jurisdictions)

    if position_values:
        person["position_held"] = position_values

    if position_jurisdictions:
        person["position_jurisdiction"] = position_jurisdictions

    # Single-valued relations.
    for field in PERSON_SINGLE_LINK_FIELDS:
        value = get_single_linked_id(g, subj, field)

        if value is not None:
            person[field] = value

    # Multi-valued relations.
    for field in PERSON_MULTI_LINK_FIELDS:
        values = unique_preserve_order(
            get_linked_ids(g, subj, field)
        )

        if values:
            person[field] = values

    if not person["_warnings"]:
        del person["_warnings"]

    return person


def extract_raw_people(g):
    """Extract all raw person records from an RDF graph."""
    return [
        extract_raw_person(g, subj)
        for subj in subjects_of_type(g, "E21_Person")
    ]


def extract_raw_place(g, subj):
    """
    Extract a raw place record from an RDF subject.

    The function preserves the source identifier, RDF types, and available
    human-readable labels. Missing labels are recorded as warnings.
    """
    place = {
        "source_id": uri_to_id(subj),
        "rdf_types": get_types(g, subj),
        "_warnings": []
    }

    labels = get_object_literals(g, subj, "P1_is_identified_by")

    if labels:
        place["labels"] = unique_preserve_order(labels)
    else:
        place["_warnings"].append("Place has no P1_is_identified_by")

    if not place["_warnings"]:
        del place["_warnings"]

    return place


def extract_raw_places(g):
    """Extract all raw place records from an RDF graph."""
    return [
        extract_raw_place(g, subj)
        for subj in subjects_of_type(g, "E53_Place")
    ]


def get_notaries(g, subj):
    """Extract notary participant identifiers from role-specific predicates."""
    return unique_preserve_order(
        get_linked_ids_by_predicate_name(
            g,
            subj,
            "P14_carried_out_byPQ14.1-Notary"
        )
    )


def get_judges(g, subj):
    """Extract judge participant identifiers from role-specific predicates."""
    return unique_preserve_order(
        get_linked_ids_by_predicate_name(
            g,
            subj,
            "P14_carried_out_byPQ14.1-Judge"
        )
    )


def extract_raw_transfer_event(g, subj):
    """
    Extract a raw transfer event from an RDF subject.

    Transfer events include source-level identifiers for transferors,
    recipients, witnesses, notaries, time metadata, and optional price
    attributes.
    """
    rdf_types = get_types(g, subj)
    event_type = (
        choose_event_type_from_rdf_types(rdf_types)
        or "GenericActivity"
    )

    event = init_raw_event(g, subj, event_type)

    price_values = get_price_values(g, subj)

    if price_values:
        event["attributes"]["price"] = price_values

    add_event_arguments(
        event,
        "transfer_from",
        get_linked_ids(g, subj, "P23_transferred_title_from")
    )

    add_event_arguments(
        event,
        "transfer_to",
        get_linked_ids(g, subj, "P22_transferred_title_to")
    )

    add_event_arguments(
        event,
        "witness",
        get_witnesses(g, subj)
    )

    add_event_arguments(
        event,
        "notary",
        get_notaries(g, subj)
    )

    return finalize_raw_event(event)


def extract_raw_document_production_event(g, subj):
    """
    Extract a raw document production event from an RDF subject.

    Document production events include notaries, judges, general participants,
    and shared event metadata such as subtype and time.
    """
    event = init_raw_event(g, subj, "DocumentProduction")

    add_event_arguments(
        event,
        "notary",
        get_notaries(g, subj)
    )

    add_event_arguments(
        event,
        "judge",
        get_judges(g, subj)
    )

    add_event_arguments(
        event,
        "participant",
        get_linked_ids(g, subj, "P14_carried_out_by")
    )

    return finalize_raw_event(event)


def extract_raw_generic_activity_event(g, subj):
    """
    Extract a raw generic activity event from an RDF subject.

    Generic activities include general participants and, when available,
    role-specific notary, judge, and witness arguments.
    """
    event = init_raw_event(g, subj, "GenericActivity")

    participants = get_linked_ids(g, subj, "P14_carried_out_by")

    add_event_arguments(event, "participant", participants)

    if has_role_label(g, subj, "witness"):
        add_event_arguments(event, "witness", participants)

    add_event_arguments(event, "notary", get_notaries(g, subj))
    add_event_arguments(event, "judge", get_judges(g, subj))

    return finalize_raw_event(event)


def extract_raw_events(g):
    """
    Extract all supported raw events from an RDF graph.

    The function processes transfer events first, then document production
    events, and finally generic activity events. A ``seen`` set prevents the
    same RDF subject from being emitted more than once when it has multiple RDF
    types.
    """
    results = []
    seen = set()

    for t in TRANSFER_EVENT_TYPES:
        for subj in subjects_of_type(g, t):
            sid = str(subj)

            if sid not in seen:
                seen.add(sid)
                results.append(
                    extract_raw_transfer_event(g, subj)
                )

    for subj in subjects_of_type(g, "E12_Production"):
        sid = str(subj)

        if sid not in seen:
            seen.add(sid)
            results.append(
                extract_raw_document_production_event(g, subj)
            )

    for subj in subjects_of_type(g, "E7_Activity"):
        sid = str(subj)

        if sid not in seen:
            seen.add(sid)
            results.append(
                extract_raw_generic_activity_event(g, subj)
            )

    return results

# JSON construction

This section converts raw RDF-derived records into the final document-level JSON
representation used for information extraction evaluation.

It builds:
- Document-level entity records preserving attested forms.
- Attribute and relation facts.
- Event records with typed arguments.
- Source-to-entity mappings used to connect RDF references to document-level
  entity identifiers.

The resulting structure is intended for IE evaluation and corpus consistency.

In [ ]:
def build_entities(raw_people, raw_places, counters):
    """
    Build document-level entity records from raw people and place records.

    The function assigns deterministic entity identifiers, preserves attested
    entity names from the source Turtle data, builds lightweight textual aliases,
    and creates a mapping from source RDF identifiers to document-level entity IDs.

    The ``canonical_name`` field should be interpreted as the preferred document-
    level display form used during corpus construction rather than as an authority-
    normalized or reconciled identifier.
    """
    entities = []
    source_to_entity_id = {}

    # People.
    for raw in raw_people:
        ent_id = counters.next_entity_id("person")

        entity = {
            "id": ent_id,
            "source_id": raw["source_id"],
            "type": "Person",
            "canonical_name": choose_canonical_name("Person", raw),
            "aliases": build_aliases("Person", raw),
        }

        entities.append(entity)
        source_to_entity_id[raw["source_id"]] = ent_id

    # Places.
    for raw in raw_places:
        ent_id = counters.next_entity_id("place")

        entity = {
            "id": ent_id,
            "source_id": raw["source_id"],
            "type": "Place",
            "canonical_name": choose_canonical_name("Place", raw),
            "aliases": build_aliases("Place", raw),
        }

        entities.append(entity)
        source_to_entity_id[raw["source_id"]] = ent_id

    return entities, source_to_entity_id


def add_fact(facts, counters, subject, predicate, obj, provenance="ttl"):
    """
    Append a document-level fact record to the fact list.

    Facts represent either literal attributes or entity-to-entity relations.
    The fact type and object type are inferred from the object value.
    """
    object_type = object_type_for_predicate_object(obj)

    fact = {
        "id": counters.next_fact_id(),
        "subject": subject,
        "predicate": predicate,
        "fact_type": "relation" if object_type == "entity" else "attribute",
        "object_type": object_type,
        "object": obj,
        "provenance": provenance,
    }

    facts.append(fact)


def build_facts(raw_people, raw_places, source_to_entity_id, counters, warnings):
    """
    Build document-level attribute and relation facts from raw person records.

    Literal fields become attribute facts. Linked source identifiers are resolved
    into document-level entity identifiers whenever possible; unresolved links are
    preserved as source identifiers and recorded as warnings.
    """
    facts = []

    for raw in raw_people:
        subj = source_to_entity_id[raw["source_id"]]

        # Literal attributes.
        for field in PERSON_LITERAL_FIELDS:
            for value in raw.get(field, []):
                add_fact(
                    facts,
                    counters,
                    subj,
                    PREDICATE_MAP[field],
                    value,
                )

        # Occupation attributes.
        for value in raw.get("occupation", []):
            add_fact(
                facts,
                counters,
                subj,
                PREDICATE_MAP["occupation"],
                value
            )

        # Occupation locations.
        for place_source_id in raw.get("occupation_location", []):
            if place_source_id in source_to_entity_id:
                obj = source_to_entity_id[place_source_id]
            else:
                obj = place_source_id
                warnings.append(
                    f"Person {raw['source_id']} occupation_location "
                    f"points to unknown place source_id={place_source_id}"
                )

            add_fact(
                facts,
                counters,
                subj,
                PREDICATE_MAP["occupation_location"],
                obj
            )

        # Position attributes.
        for value in raw.get("position_held", []):
            add_fact(
                facts,
                counters,
                subj,
                PREDICATE_MAP["position_held"],
                value
            )

        # Position jurisdictions.
        for place_source_id in raw.get("position_jurisdiction", []):
            if place_source_id in source_to_entity_id:
                obj = source_to_entity_id[place_source_id]
            else:
                obj = place_source_id
                warnings.append(
                    f"Person {raw['source_id']} position_jurisdiction "
                    f"points to unknown place source_id={place_source_id}"
                )

            add_fact(
                facts,
                counters,
                subj,
                PREDICATE_MAP["position_jurisdiction"],
                obj
            )

        # Single-valued relations.
        for field in PERSON_SINGLE_LINK_FIELDS:
            if field not in raw:
                continue

            target_source_id = raw[field]

            if target_source_id in source_to_entity_id:
                obj = source_to_entity_id[target_source_id]
            else:
                obj = target_source_id
                warnings.append(
                    f"Person {raw['source_id']} field {field} "
                    f"points to unknown source_id={target_source_id}"
                )

            add_fact(
                facts,
                counters,
                subj,
                PREDICATE_MAP[field],
                obj,
            )

        # Multi-valued relations.
        for field in PERSON_MULTI_LINK_FIELDS:
            for target_source_id in raw.get(field, []):
                if target_source_id in source_to_entity_id:
                    obj = source_to_entity_id[target_source_id]
                else:
                    obj = target_source_id
                    warnings.append(
                        f"Person {raw['source_id']} field {field} "
                        f"points to unknown source_id={target_source_id}"
                    )

                add_fact(
                    facts,
                    counters,
                    subj,
                    PREDICATE_MAP[field],
                    obj,
                )

    return facts


def build_events(raw_events, source_to_entity_id, counters, warnings):
    """
    Build document-level event records from raw event records.

    Event arguments are resolved from source RDF identifiers to document-level entity
    identifiers. Unresolved arguments are preserved under
    ``unresolved_source_id`` and recorded as warnings.
    """
    events = []

    for raw in raw_events:
        event = {
            "id": counters.next_event_id(),
            "source_id": raw["source_id"],
            "type": raw["type"],
            "attributes": raw.get("attributes", {}),
            "arguments": [],
            "provenance": "ttl",
        }

        if raw.get("subtype"):
            event["subtype"] = raw["subtype"]

        for arg in raw.get("arguments", []):
            source_entity_id = arg["source_entity_id"]

            if source_entity_id in source_to_entity_id:
                event["arguments"].append({
                    "role": arg["role"],
                    "entity": source_to_entity_id[source_entity_id]
                })

            else:
                event["arguments"].append({
                    "role": arg["role"],
                    "unresolved_source_id": source_entity_id
                })

                warnings.append(
                    f"Event {raw['source_id']} argument role={arg['role']} "
                    f"unresolved source_entity_id={source_entity_id}"
                )

        if raw.get("_warnings"):
            event["warnings"] = raw["_warnings"]

        events.append(event)

    return events

# NER-based entity completion

This section uses NER annotations from the original dataset to complete the
document-level entity inventory.

The Turtle graph is treated as the primary source. NER annotations are only used
to add person and place entities that are present in the dataset but not
materialized in the Turtle graph.

NER mentions are not stored as a separate output layer. They are used only as an
input signal for completing the entity inventory.

Entity names are preserved in their attested form. No authority normalization,
cross-document entity reconciliation, or canonical name standardization is
performed.

In [ ]:
def build_alias_index(entities):
    """
    Build a lightweight index of attested entity forms.

    The index is used only to detect whether a NER mention is already represented
    in the document-level entity inventory. It does not imply authority control,
    canonical normalization, or external entity linking.
    """
    alias_index = defaultdict(list)

    for ent in entities:
        for alias in ent.get("aliases", []):
            alias_index[normalize_text(alias)].append(ent)

        alias_index[normalize_text(ent["canonical_name"])].append(ent)

    return alias_index


def resolve_mention_to_entity(mention, entities, alias_index):
    """
    Check whether a NER-derived attested name is already represented by an entity.

    Matching is used only to avoid duplicate document-level entities when
    importing NER-derived records. It is not stored as a separate mention
    alignment layer and does not imply authority reconciliation or entity
    normalization.
    """
    mention_text = mention.get("name", "")
    mention_label = mention.get("label")
    mention_type = NER_LABEL_TO_ENTITY_TYPE.get(mention_label)

    if not mention_type:
        return None, "unsupported_label"

    norm = normalize_text(mention_text)

    if not norm:
        return None, "empty_mention"

    exact_candidates = [
        ent
        for ent in alias_index.get(norm, [])
        if ent["type"] == mention_type
    ]

    exact_candidates = list({
        e["id"]: e
        for e in exact_candidates
    }.values())

    if len(exact_candidates) == 1:
        return exact_candidates[0]["id"], "alias_exact"

    if len(exact_candidates) > 1:
        return None, "ambiguous_exact"

    soft_candidates = []

    for ent in entities:
        if ent["type"] != mention_type:
            continue

        for alias in ent.get("aliases", []):
            norm_alias = normalize_text(alias)

            if not norm_alias:
                continue

            if (
                norm == norm_alias
                or norm in norm_alias
                or norm_alias in norm
            ):
                soft_candidates.append(ent)
                break

    soft_candidates = list({
        e["id"]: e
        for e in soft_candidates
    }.values())

    if len(soft_candidates) == 1:
        return soft_candidates[0]["id"], "alias_soft"

    if len(soft_candidates) > 1:
        return None, "ambiguous_soft"

    return None, "no_match"


def complete_entities_from_ner(doc_id, ner_data, entities, counters, warnings):
    """
    Complete the entity inventory using NER annotations from the original dataset.

    The Turtle graph remains the primary source of structured entities. NER
    annotations are used only to add person or place entities whose attested names
    are not already represented among the RDF-derived entities.

    Added entities preserve their attested form and are marked with
    ``provenance = "ner"``. The function does not perform authority normalization,
    cross-document reconciliation, or canonical name standardization.
    """
    if not ner_data:
        return entities

    possible_keys = [
        f"output_BIO_{doc_id}.txt",
        f"{doc_id}.txt",
        doc_id,
    ]

    doc_ner = None

    for key in possible_keys:
        if key in ner_data:
            doc_ner = ner_data[key]
            break

    if doc_ner is None:
        return entities

    alias_index = build_alias_index(entities)
    items = doc_ner.get("entities", [])

    for item in items:
        mention_text = item.get("name", "")
        mention_label = item.get("label")
        entity_type = NER_LABEL_TO_ENTITY_TYPE.get(mention_label)

        if not entity_type:
            continue

        entity_id, method = resolve_mention_to_entity(
            item,
            entities,
            alias_index
        )

        if entity_id is not None:
            continue

        ent_id = counters.next_entity_id(entity_type.lower())

        new_entity = {
            "id": ent_id,
            "source_id": f"ner:{doc_id}:{item.get('start')}:{item.get('end')}",
            "type": entity_type,
            "canonical_name": mention_text,
            "aliases": [mention_text],
            "provenance": "ner",
        }

        entities.append(new_entity)
        alias_index[normalize_text(mention_text)].append(new_entity)

    return entities

# Full document transformation

This section orchestrates the complete transformation of one Turtle document.

For each document, it parses the RDF graph, extracts raw records, builds
normalized entities, facts, events, and returns a single
document-level JSON object with warnings.

In [ ]:
def collect_raw_warnings(raw_people, raw_places, raw_events):
    """
    Collect warnings generated during raw RDF extraction.

    Each warning is prefixed with the source identifier of the record that
    produced it, making downstream inspection and debugging easier.
    """
    warnings = []

    for bucket in (raw_people, raw_places, raw_events):
        for item in bucket:
            for w in item.get("_warnings", []):
                sid = item.get("source_id", "UNKNOWN")
                warnings.append(f"{sid}: {w}")

    return warnings


def transform_document(ttl_path, ner_data=None):
    """
    Transform one Turtle document into a document-level JSON object.

    The transformation includes RDF parsing, raw extraction, entity construction,
    NER-based entity completion, fact generation, event normalization, and
    warning aggregation.

    Entity names are preserved in their attested form. NER annotations are used only
    to complete the entity inventory when the original dataset contains attested
    entities not materialized in the Turtle graph.
    """
    doc_id = Path(ttl_path).stem
    g = parse_ttl(ttl_path)

    raw_people = extract_raw_people(g)
    raw_places = extract_raw_places(g)
    raw_events = extract_raw_events(g)

    warnings = collect_raw_warnings(
        raw_people,
        raw_places,
        raw_events
    )

    counters = Counters()

    entities, source_to_entity_id = build_entities(
        raw_people,
        raw_places,
        counters
    )

    entities = complete_entities_from_ner(
        doc_id,
        ner_data or {},
        entities,
        counters,
        warnings
    )

    facts = build_facts(
        raw_people,
        raw_places,
        source_to_entity_id,
        counters,
        warnings
    )

    events = build_events(
        raw_events,
        source_to_entity_id,
        counters,
        warnings
    )

    output = {
        "document_id": doc_id,
        "entities": entities,
        "facts": facts,
        "events": events,
        "warnings": unique_preserve_order(warnings),
    }

    return output

# Saving and corpus summary

This section defines output utilities for writing document-level JSON files and
computing aggregate corpus statistics across all processed documents.

In [ ]:
def save_document_json(doc_json, out_dir):
    """
    Save a normalized document JSON object to disk.

    Each transformed document is stored as an individual JSON file named after
    the document identifier.
    """
    out_path = Path(out_dir) / f"{doc_json['document_id']}.json"

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(
            doc_json,
            f,
            ensure_ascii=False,
            indent=2
        )

    return out_path


def summarize_corpus(all_docs):
    """
    Compute aggregate statistics for the generated corpus.

    The summary includes document counts and the total number of entities,
    facts, events, and documents containing warnings.
    """
    return {
        "n_documents": len(all_docs),

        "total_entities": sum(
            len(d["entities"])
            for d in all_docs
        ),

        "total_facts": sum(
            len(d["facts"])
            for d in all_docs
        ),

        "total_events": sum(
            len(d["events"])
            for d in all_docs
        ),

        "documents_with_warnings": sum(
            1
            for d in all_docs
            if d["warnings"]
        ),
    }

# Batch execution

This section runs the full transformation pipeline over all Turtle files in the
configured input directory.

Each document is transformed independently, saved as JSON, and appended to a
corpus-level JSONL file.

In [ ]:
ner_data = load_ner_json(NER_PATH)

all_docs = []

for ttl_path in tqdm(sorted(TTL_DIR.glob("*.ttl"))):
    try:
        doc_json = transform_document(ttl_path, ner_data=ner_data)
        save_document_json(doc_json, JSON_DIR)
        all_docs.append(doc_json)
    except Exception as e:
        print(f"Error en {ttl_path.name}: {e}")

with open(JSONL_PATH, "w", encoding="utf-8") as f:
    for doc in all_docs:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

summary = summarize_corpus(all_docs)

print(f"Guardado JSONL en {JSONL_PATH}")
print(f"Procesados {len(all_docs)} documentos")
print(json.dumps(summary, ensure_ascii=False, indent=2))

# Output inspection

This section prints a truncated example of the first generated document so that
the structure of the normalized JSON output can be inspected directly in Colab.

In [ ]:
if all_docs:
    print("\nEjemplo primer documento:")
    print(json.dumps(all_docs[0], ensure_ascii=False, indent=2)[:4000])